In [ ]:
!pip uninstall -y sentence-transformers transformers huggingface-hub
!pip install sentence-transformers==2.2.2 transformers==4.30.2 huggingface-hub==0.16.4
!pip install sentence-transformers scikit-learn pandas numpy tqdm

Found existing installation: sentence-transformers 5.1.2
Uninstalling sentence-transformers-5.1.2:
  Successfully uninstalled sentence-transformers-5.1.2
Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
Found existing installation: huggingface-hub 0.36.0
Uninstalling huggingface-hub-0.36.0:
  Successfully uninstalled huggingface-hub-0.36.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 10.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 18.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 26.0 MB/s eta 0:00:00


In [ ]:
# SETUP

import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import RidgeCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import pickle

# Load Data
df = pd.read_csv("VAD.csv", encoding="utf-8", engine="python", on_bad_lines="skip")

# Standardize column names
df.columns = [c.strip().lower() for c in df.columns]

# Expecting: text, v, a, d
expected_cols = {"text", "v", "a", "d"}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Your VAD.csv is missing columns: {missing}")

# Rename into valence / arousal / dominance for clarity
df = df.rename(columns={
    "v": "valence",
    "a": "arousal",
    "d": "dominance"
})

# Drop rows missing text or scores
df = df.dropna(subset=["text", "valence", "arousal", "dominance"])

# Ensure VAD scores are floats
df["valence"] = df["valence"].astype(float)
df["arousal"] = df["arousal"].astype(float)
df["dominance"] = df["dominance"].astype(float)

print(f"Loaded {len(df):,} VAD samples.")
print(df.head(), "\n")

# --- Quick summary ---
print("Dataset Summary")
print(df.describe())



Loaded 310,538 VAD samples.
             id  split emotion_label  valence  arousal  dominance  \
0  a0f805e3-e15    dev       fearful     2.39     6.78       2.54   
1  a415e639-4db   test       neutral     4.09     4.84       5.37   
2  ebf2507e-847  train         angry     2.81     8.63       5.95   
3  360d8c31-6c9  train       disgust     2.68     6.94       3.52   
4  fc3fd738-176   test      surprise     6.95     5.83       5.89   

                                                text  
0  I felt pressured by my own anxiety., and I was...  
1  I simply experienced it and continued., and it...  
2  I reacted harshly because I felt overwhelmed.,...  
3  I felt violated by how disturbing it was., in ...  
4  I tried to make sense of the sudden turn., tha...   

Dataset Summary
             valence        arousal      dominance
count  310538.000000  310538.000000  310538.000000
mean        4.428955       5.556738       5.096671
std         2.304871       1.791122       1.861135
min  

In [ ]:


# Embedding

model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)
data = df
# Encode sentences to fixed-length embeddings

tqdm.pandas()
X = np.vstack(data["text"].progress_apply(lambda x: embedder.encode(str(x), show_progress_bar=False)))
y = data[["valence", "arousal", "dominance"]].values

# Training Split 85/15

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Training

reg = MultiOutputRegressor(RidgeCV(alphas=(0.1, 1.0, 10.0)))
reg.fit(X_train, y_train)

# Evaluation

preds = reg.predict(X_test)
mae = mean_absolute_error(y_test, preds, multioutput='raw_values')
r2 = r2_score(y_test, preds, multioutput='raw_values')

print("\nEvaluation Metrics")
print(f"MAE - Valence: {mae[0]:.3f}, Arousal: {mae[1]:.3f}, Dominance: {mae[2]:.3f}")
print(f"R²  - Valence: {r2[0]:.3f}, Arousal: {r2[1]:.3f}, Dominance: {r2[2]:.3f}")

#  SAVE MODEL

with open("vad_regression_model.pkl", "wb") as f:
    pickle.dump({"embedder": embedder, "model": reg}, f)

print("\n Model saved as vad_regression_model.pkl")

#  SAMPLE PREDICTION

sample = "I feel nervous but excited for tomorrow."
vec = embedder.encode(sample)
pred = reg.predict([vec])[0]
print(f"\nUser input: {sample}")
print(f"Predicted VAD → Valence: {pred[0]:.2f}, Arousal: {pred[1]:.2f}, Dominance: {pred[2]:.2f}")


100%|██████████| 310538/310538 [30:28<00:00, 169.85it/s]



Evaluation Metrics
MAE - Valence: 0.868, Arousal: 0.851, Dominance: 0.890
R²  - Valence: 0.773, Arousal: 0.655, Dominance: 0.648

 Model saved as vad_regression_model.pkl

User input: I feel nervous but excited for tomorrow.
Predicted VAD → Valence: 6.42, Arousal: 6.35, Dominance: 5.94


In [ ]:
# INTERACTIVE VAD PREDICTOR

import pickle
from sentence_transformers import SentenceTransformer


In [ ]:
# Load the trained model (if not already in memory)

with open("vad_regression_model.pkl", "rb") as f:
    bundle = pickle.load(f)

embedder = bundle["embedder"]
model = bundle["model"]

print("VAD model loaded and ready.")

VAD model loaded and ready.


In [ ]:
# Create a text input box (Colab magic)

from IPython.display import display
import ipywidgets as widgets
import numpy as np

prompt_box = widgets.Textarea(
    placeholder='Type something to analyze emotion (e.g., "I’m feeling hopeful today.")',
    description='Input:',
    layout=widgets.Layout(width='100%', height='80px')
)
output_box = widgets.Output()

button = widgets.Button(description="Predict VAD", button_style='info')

def cap_scale(pred, low=1.0, high=10.0):
    """Clip model predictions into the desired VAD range."""
    return np.clip(pred, low, high)

def on_button_click(b):
    text = prompt_box.value.strip()
    if not text:
        with output_box:
            output_box.clear_output()
            print("Please enter some text.")
        return

    # Encode and predict
    emb = embedder.encode(text)
    raw_pred = model.predict([emb])[0]

    pred = cap_scale(raw_pred, 1.0, 10.0)

    # Display results
    with output_box:
        output_box.clear_output()
        print(f"Input: {text}\n")
        print(f"🔹 Valence (Pleasantness): {pred[0]:.3f}")
        print(f"🔹 Arousal (Energy):       {pred[1]:.3f}")
        print(f"🔹 Dominance (Control):    {pred[2]:.3f}")

button.on_click(on_button_click)

display(prompt_box, button, output_box)



Textarea(value='', description='Input:', layout=Layout(height='80px', width='100%'), placeholder='Type somethi…

Button(button_style='info', description='Predict VAD', style=ButtonStyle())

Output()